# London road safety EDA
Quick sanity check on the cleaned STATS19 + AADT data before we feed it into clustering / ML.

Inputs: `data/processed/london_accidents.parquet`, `data/processed/london_aadt.parquet` (both produced by `app.data.preprocessing`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROCESSED = Path('..').resolve().parent / 'data' / 'processed'
collisions = pd.read_parquet(PROCESSED / 'london_accidents.parquet')
aadt = pd.read_parquet(PROCESSED / 'london_aadt.parquet')
print(f'collisions: {len(collisions):,} rows, years {collisions.collision_year.min()}-{collisions.collision_year.max()}')
print(f'aadt: {len(aadt):,} count points')

## Severity distribution
Slight dominates as expected. The fatal count is small but absolute risk per fatal incident is high — we'll weight by severity in risk scoring.

In [ ]:
sev = collisions['collision_severity_label'].value_counts().reindex(['Slight', 'Serious', 'Fatal'])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(sev.index, sev.values, color=['#1f77b4', '#ff7f0e', '#d62728'])
ax.set_ylabel('collisions')
ax.set_title('London collisions by severity (2020-2024)')
for i, v in enumerate(sev.values):
    ax.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.show()
sev

## When do collisions happen?
Hour-of-day shows the morning commute peak (~8am) and a bigger PM peak (4-6pm). Weekday patterns differ from weekends.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

hourly = collisions['hour'].value_counts().sort_index()
axes[0].bar(hourly.index, hourly.values, color='#1f77b4')
axes[0].set_xlabel('hour of day')
axes[0].set_ylabel('collisions')
axes[0].set_title('hour-of-day distribution')
axes[0].set_xticks(range(0, 24, 2))

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = collisions['day_of_week_label'].value_counts().reindex(dow_order)
axes[1].bar(dow.index, dow.values, color='#2ca02c')
axes[1].set_ylabel('collisions')
axes[1].set_title('day-of-week distribution')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Weather conditions
Most collisions happen in fine weather — that's not an indictment of fine weather, just a reflection of how often it's the prevailing condition. We'll let the Random Forest sort out the conditional effect.

In [ ]:
weather = collisions['weather_conditions_label'].value_counts().head(6)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(weather.index[::-1], weather.values[::-1], color='#9467bd')
ax.set_xlabel('collisions')
ax.set_title('weather at time of collision (top 6)')
plt.tight_layout()
plt.show()
weather

## Where do they happen?
Scatter of every collision over the London bbox. Density alone reveals the major arteries — A40, A406 (North Circular), A205 (South Circular), the City.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(
    collisions['longitude'], collisions['latitude'],
    s=1, alpha=0.08, color='#d62728',
)
ax.set_xlim(-0.51, 0.33)
ax.set_ylim(51.28, 51.69)
ax.set_aspect('equal')
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
ax.set_title(f'{len(collisions):,} collisions, london bbox')
plt.show()

## AADT (traffic volume) distribution
Long-tailed — most road segments carry modest traffic, a handful carry over 100k vehicles a day. Log scale helps.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(aadt['all_motor_vehicles'].clip(lower=1), bins=50, color='#17becf')
ax.set_xscale('log')
ax.set_xlabel('all_motor_vehicles (log scale)')
ax.set_ylabel('count points')
ax.set_title(f'london AADT distribution ({len(aadt):,} count points)')
plt.tight_layout()
plt.show()
aadt['all_motor_vehicles'].describe().astype(int)